In [1]:
import pandas as pd

In [2]:
%%time
data = pd.read_csv('barra2-ob53.csv.gz', compression='gzip')


CPU times: user 8.08 s, sys: 822 ms, total: 8.9 s
Wall time: 8.9 s


In [3]:
variable_id = [
    'tas',
    'huss',
    'ps',
    'uas',
    'vas',
    'rsds',
    'rsus',
    'rlds',
    'rlus'
]
file_type = 'f'                       # U=2
project_id = 'output'                 # U=2
activity_id = 'reanalysis'            # U=1
RCM_institution_id = 'BOM'            # U=1
driving_experiment_id = 'historical'  # U=1
freq = '1hr'                          # U=6
version_realisation = 'v1'            # U=2


In [4]:
### BARRA C2 4.4 KM
domain_id = 'AUST-04'                 # U=6
source_id = 'BARRA-C2'                # U=3


BARRAC2 = data[
    (data['file_type'] == file_type) &
    (data['project_id'] == project_id) &
    (data['activity_id'] == activity_id) &
    (data['domain_id'] == domain_id) &
    (data['source_id'] == source_id) &
    (data['RCM_institution_id'] == RCM_institution_id) &
    (data['driving_experiment_id'] == driving_experiment_id) &
    (data['version_realisation'] == version_realisation) &
    (data['freq'] == freq) &
    (data['variable_id'].isin(variable_id))
].copy()

BARRAC2['start_time'] = BARRAC2['start_time'].astype(int)
BARRAC2['year'] = BARRAC2['start_time'].astype(str).str[:4].astype(int)
BARRAC2['month'] = BARRAC2['start_time'].astype(str).str[-2:].astype(int)

BARRAC2 = (
    BARRAC2[['year', 'month', 'variable_id', 'path']]
    .sort_values(by=['year', 'month', 'variable_id'])
    .reset_index(drop=True)
)


In [5]:
### BARRA R2 12 KM
domain_id = 'AUST-11'
source_id = 'BARRA-R2'


BARRAR2 = data[
    (data['file_type'] == file_type) &
    (data['project_id'] == project_id) &
    (data['activity_id'] == activity_id) &
    (data['source_id'] == source_id) &
    (data['RCM_institution_id'] == RCM_institution_id) &
    (data['driving_experiment_id'] == driving_experiment_id) &
    (data['freq'] == freq) &
    (data['variable_id'].isin(variable_id))
].copy()

BARRAR2['start_time'] = BARRAR2['start_time'].astype(int)
BARRAR2['year'] = BARRAR2['start_time'].astype(str).str[:4].astype(int)
BARRAR2['month'] = BARRAR2['start_time'].astype(str).str[-2:].astype(int)

BARRAR2 = (
    BARRAR2[['year', 'month', 'variable_id', 'path']]
    .sort_values(by=['year', 'month', 'variable_id'])
    .reset_index(drop=True)
)


In [6]:
### BARRA R2 22 KM
domain_id = 'AUST-22'
source_id = 'BARRA-RE2'


BARRARE2 = data[
    (data['file_type'] == file_type) &
    (data['project_id'] == project_id) &
    (data['activity_id'] == activity_id) &
    (data['source_id'] == source_id) &
    (data['RCM_institution_id'] == RCM_institution_id) &
    (data['driving_experiment_id'] == driving_experiment_id) &
    (data['freq'] == freq) &
    (data['variable_id'].isin(variable_id))
].copy()

BARRARE2['start_time'] = BARRARE2['start_time'].astype(int)
BARRARE2['year'] = BARRARE2['start_time'].astype(str).str[:4].astype(int)
BARRARE2['month'] = BARRARE2['start_time'].astype(str).str[-2:].astype(int)

BARRARE2 = (
    BARRARE2[['year', 'month', 'variable_id', 'path']]
    .sort_values(by=['year', 'month', 'variable_id'])
    .reset_index(drop=True)
)


In [7]:
%%time
ERA5 = pd.read_csv('era5-rt52.csv.gz', compression='gzip')

product = 'era5-reanalysis'
stream  = 'oper'
levtype = 'sfc'

ERA5_VARS = [
    "2t",        # 2 m air temperature
    "2d",        # 2 m dew-point temperature
    "sp",        # surface pressure
    "10u",       # 10 m U wind
    "10v",       # 10 m V wind
    "msdwswrf",  # mean surface downward shortwave radiation flux
    "msnswrf",   # mean surface net shortwave radiation flux
    "msdwlwrf",  # mean surface downward longwave radiation flux
    "msnlwrf",   # mean surface net longwave radiation flux
]

ERA5 = ERA5[
    (ERA5['file_type'] == file_type) &
    (ERA5['product'] == product) &
    (ERA5['stream'] == stream) &
    (ERA5['levtype'] == levtype) &
    (ERA5['variable'].isin(ERA5_VARS))
]

ERA5['start_date'] = pd.to_datetime(
    ERA5['time_range'].astype(str).str.split('-').str[0],
    format='%Y%m%d'
)

ERA5['end_date'] = pd.to_datetime(
    ERA5['time_range'].astype(str).str.split('-').str[1],
    format='%Y%m%d'
)

ERA5['year'] = ERA5['start_date'].dt.year
ERA5 = ERA5[ERA5['year'] >= BARRAC2['year'].min()]
ERA5 = ERA5[ERA5['year'] <= BARRAC2['year'].max()]
ERA5['month'] = ERA5['start_date'].dt.month

ERA5 = (
    ERA5[['year', 'month', 'variable', 'path']]
    .sort_values(by=['year', 'month', 'variable'])
    .reset_index(drop=True)
)


CPU times: user 2.05 s, sys: 70.1 ms, total: 2.12 s
Wall time: 2.11 s


In [8]:
"""
ERA5: global
└── BARRA-RE2: large regional domain
      └── BARRA-R2: smaller high-resolution regional domain
            └── BARRA-C2: smaller high-resolution km-scale Australian domain
"""

BARRAC2_LTLN = {
    'lat_min': -45.69,
    'lat_max': -5.01,
    'lon_min': 108.02,
    'lon_max': 159.9,
    'dim': {'lat': 1018, 'lon': 1298}
}

BARRAR2_LTLN = {
    'lat_min': -57.97,
    'lat_max': 12.98,
    'lon_min': 88.48,
    'lon_max': 207.39,
    'dim': {'lat': 646, 'lon': 1082}
}

BARRARE2_LTLN = {
    'lat_min': -56.49,
    'lat_max': 11.71,
    'lon_min': 89.53,
    'lon_max': 206.13,
    'dim': {'lat': 311, 'lon': 531}
}

ERA5_LTLN = {
    'lat_min': -90.0,
    'lat_max': 90.0,
    'lon_min': -180.0,
    'lon_max': 179.75,
    'dim': {'latitude': 721, 'longitude': 1440}
}



In [9]:
%%writefile nc_time_worker.py

import pandas as pd
import xarray as xr


def extract_file_datetimes(path):
    try:
        with xr.open_dataset(
            path,
            decode_times=True,
            cache=False
        ) as ds:

            times = pd.to_datetime(ds["time"].values)

        return path, times, None

    except Exception as e:
        return path, None, repr(e)
        

Overwriting nc_time_worker.py


In [11]:
import multiprocessing as mp

from tqdm.auto import tqdm
from nc_time_worker import extract_file_datetimes

def expand_datetime_parallel(
    df,
    variable_col,
    n_workers=12
):

    paths = df["path"].drop_duplicates().tolist()

    print(f"Files to process: {len(paths):,}")
    print(f"Processes: {n_workers}")

    ctx = mp.get_context("spawn")

    time_map = {}
    errors = []

    with ctx.Pool(processes=n_workers) as pool:

        iterator = pool.imap_unordered(
            extract_file_datetimes,
            paths,
            chunksize=1
        )

        for path, times, error in tqdm(
            iterator,
            total=len(paths),
            desc="Reading datetimes"
        ):

            if error is None:
                time_map[path] = times
            else:
                errors.append((path, error))

    # Keep ONLY variable + path from original catalogue
    out = df[
        [variable_col, "path"]
    ].copy()

    # Standard name for all datasets
    out = out.rename(
        columns={variable_col: "variable"}
    )

    # Attach actual datetime array from each NetCDF
    out["datetime"] = out["path"].map(time_map)

    # Remove files that failed
    out = out[
        out["datetime"].notna()
    ].copy()

    # One row per hourly timestamp
    out = out.explode(
        "datetime",
        ignore_index=True
    )

    out["datetime"] = pd.to_datetime(
        out["datetime"]
    )

    # Final column order
    out = out[
        [
            "datetime",
            "variable",
            "path"
        ]
    ]

    out = out.sort_values(
        ["datetime", "variable"]
    ).reset_index(drop=True)

    return out, errors


In [12]:
%%time
BARRAC2, ERRORS = expand_datetime_parallel(
    BARRAC2,
    variable_col="variable_id",
    n_workers=12
)
BARRAC2.to_csv('BARRAC2.csv.gz', index=False, compression='gzip')


Files to process: 5,103
Processes: 12


Reading datetimes:   0%|          | 0/5103 [00:00<?, ?it/s]

CPU times: user 11.5 s, sys: 1.45 s, total: 13 s
Wall time: 22.4 s


In [15]:
%%time
BARRAR2, ERRORS = expand_datetime_parallel(
    BARRAR2,
    variable_col="variable_id",
    n_workers=12
)
BARRAR2.to_csv('BARRAR2.csv.gz', index=False, compression='gzip')


Files to process: 5,103
Processes: 12


Reading datetimes:   0%|          | 0/5103 [00:00<?, ?it/s]

CPU times: user 33.7 s, sys: 1.35 s, total: 35.1 s
Wall time: 44.5 s


In [16]:
%%time
BARRARE2, ERRORS = expand_datetime_parallel(
    BARRARE2,
    variable_col="variable_id",
    n_workers=12
)
BARRARE2.to_csv('BARRARE2.csv.gz', index=False, compression='gzip')


Files to process: 5,103
Processes: 12


Reading datetimes:   0%|          | 0/5103 [00:00<?, ?it/s]

CPU times: user 34.7 s, sys: 1.42 s, total: 36.2 s
Wall time: 46 s


In [17]:
%%time
ERA5, ERRORS = expand_datetime_parallel(
    ERA5,
    variable_col="variable",
    n_workers=12
)
ERA5.to_csv('ERA5.csv.gz', index=False, compression='gzip')


Files to process: 5,103
Processes: 12


Reading datetimes:   0%|          | 0/5103 [00:00<?, ?it/s]

CPU times: user 30.4 s, sys: 1.27 s, total: 31.6 s
Wall time: 40.1 s
